In [3]:
import requests
import pandas as pd
import os
import math
import time
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

os.makedirs('data/aerial_images', exist_ok=True)
lsoas = pd.read_csv('data/test_lsoas.csv')

ZOOM = 17
TILE_SIZE = 256
HEADERS = {'User-Agent': 'urban-inequality-research/1.0 sarah.hong.25@ucl.ac.uk'}

def lat_lon_to_tile(lat, lon, zoom):
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    lat_rad = math.radians(lat)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1.0 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y

def download_osm_tile(x, y, zoom):
    url = f"https://tile.openstreetmap.org/{zoom}/{x}/{y}.png"
    r = requests.get(url, headers=HEADERS, timeout=10)
    if r.status_code == 200:
        return Image.open(BytesIO(r.content)).convert('RGB')
    return None

def download_lsoa_aerial(lat, lon, zoom=ZOOM, grid=3):
    cx, cy = lat_lon_to_tile(lat, lon, zoom)
    half = grid // 2
    stitched = Image.new('RGB', (TILE_SIZE * grid, TILE_SIZE * grid))
    for dy in range(grid):
        for dx in range(grid):
            tile = download_osm_tile(cx - half + dx, cy - half + dy, zoom)
            if tile:
                stitched.paste(tile, (dx * TILE_SIZE, dy * TILE_SIZE))
    return stitched

print(f'Ready to download aerial images for {len(lsoas)} LSOAs')

Ready to download aerial images for 2000 LSOAs


In [4]:
success = 0
failed = 0

for idx, row in lsoas.iterrows():
    lsoa_code = row['LSOA11CD']
    out_path = f"data/aerial_images/{lsoa_code}.jpg"
    if os.path.exists(out_path):
        success += 1
        continue
    try:
        img = download_lsoa_aerial(row['lat'], row['lon'])
        img.save(out_path, quality=90)
        success += 1
    except Exception as e:
        failed += 1
    time.sleep(0.5)
    if (idx + 1) % 50 == 0:
        print(f'Progress: {idx+1}/{len(lsoas)} | Success: {success} | Failed: {failed}')

print(f'\nDone! {success} downloaded, {failed} failed')

Progress: 100/2000 | Success: 100 | Failed: 0
Progress: 150/2000 | Success: 150 | Failed: 0
Progress: 200/2000 | Success: 200 | Failed: 0
Progress: 300/2000 | Success: 300 | Failed: 0
Progress: 350/2000 | Success: 350 | Failed: 0
Progress: 400/2000 | Success: 400 | Failed: 0
Progress: 500/2000 | Success: 500 | Failed: 0
Progress: 550/2000 | Success: 550 | Failed: 0
Progress: 600/2000 | Success: 600 | Failed: 0
Progress: 700/2000 | Success: 700 | Failed: 0
Progress: 750/2000 | Success: 750 | Failed: 0
Progress: 800/2000 | Success: 800 | Failed: 0
Progress: 900/2000 | Success: 900 | Failed: 0
Progress: 950/2000 | Success: 950 | Failed: 0
Progress: 1000/2000 | Success: 1000 | Failed: 0
Progress: 1100/2000 | Success: 1100 | Failed: 0
Progress: 1150/2000 | Success: 1150 | Failed: 0
Progress: 1200/2000 | Success: 1200 | Failed: 0
Progress: 1300/2000 | Success: 1300 | Failed: 0
Progress: 1350/2000 | Success: 1350 | Failed: 0
Progress: 1400/2000 | Success: 1400 | Failed: 0
Progress: 1500/2000 